In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
pd.read_csv("data/raw/ab_data.csv").to_sql("raw", conn, index=False)
pd.read_csv("data/clean/ab_clean.csv").to_sql("ab", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

In [2]:
q("""
SELECT 'Wrong page for group' AS check_name,
       SUM(CASE WHEN (grp_ctrl = 1 AND page_old = 0) OR (grp_ctrl = 0 AND page_old = 1) THEN 1 ELSE 0 END) AS rows_affected
FROM (SELECT ("group" = 'control') AS grp_ctrl, (landing_page = 'old_page') AS page_old FROM raw)
UNION ALL
SELECT 'Users appearing more than once', COUNT(*)
FROM (SELECT user_id FROM raw GROUP BY user_id HAVING COUNT(*) > 1)
""")

,check_name,rows_affected
0,Wrong page for group,3893
1,Users appearing more than once,3894


In [3]:
q("""
SELECT "group",
       COUNT(*) AS users,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share_pct,
       SUM(converted) AS conversions,
       ROUND(100.0 * AVG(converted), 3) AS conversion_pct
FROM ab
GROUP BY "group"
""")

,group,users,share_pct,conversions,conversion_pct
0,control,145274,49.994,17489,12.039
1,treatment,145310,50.006,17264,11.881


In [4]:
q("""
WITH daily AS (
    SELECT date, "group", COUNT(*) AS users, SUM(converted) AS conv
    FROM ab GROUP BY date, "group")
SELECT date, "group",
       SUM(users) OVER (PARTITION BY "group" ORDER BY date) AS cum_users,
       ROUND(100.0 * SUM(conv) OVER (PARTITION BY "group" ORDER BY date)
                   / SUM(users) OVER (PARTITION BY "group" ORDER BY date), 3) AS cum_conversion_pct
FROM daily
ORDER BY date, "group"
""").head(10)

,date,group,cum_users,cum_conversion_pct
0,2017-01-02,control,2859,12.557
1,2017-01-02,treatment,2853,11.987
2,2017-01-03,control,9449,11.737
3,2017-01-03,treatment,9471,11.562
4,2017-01-04,control,16027,11.924
5,2017-01-04,treatment,16012,11.604
6,2017-01-05,control,22454,12.038
7,2017-01-05,treatment,22517,11.573
8,2017-01-06,control,29060,11.924
9,2017-01-06,treatment,29264,11.752


In [5]:
q("""
SELECT CASE strftime('%w', date)
         WHEN '0' THEN 'Sun' WHEN '1' THEN 'Mon' WHEN '2' THEN 'Tue' WHEN '3' THEN 'Wed'
         WHEN '4' THEN 'Thu' WHEN '5' THEN 'Fri' ELSE 'Sat' END AS weekday,
       ROUND(100.0 * AVG(CASE WHEN "group" = 'control' THEN converted END), 3) AS control_pct,
       ROUND(100.0 * AVG(CASE WHEN "group" = 'treatment' THEN converted END), 3) AS treatment_pct,
       ROUND(100.0 * (AVG(CASE WHEN "group" = 'treatment' THEN converted END)
                    - AVG(CASE WHEN "group" = 'control' THEN converted END)), 3) AS diff_pp
FROM ab
GROUP BY strftime('%w', date)
ORDER BY strftime('%w', date)
""")

,weekday,control_pct,treatment_pct,diff_pp
0,Sun,11.952,11.743,-0.209
1,Mon,12.280,11.949,-0.330
2,Tue,11.675,12.234,0.559
3,Wed,12.184,11.884,-0.300
4,Thu,12.173,11.821,-0.352
5,Fri,11.583,11.754,0.170
6,Sat,12.457,11.706,-0.751
